## Imports

In [0]:
from datetime import date, datetime
from pyspark.sql import functions as F
from pyspark.sql.functions import lower, col, regexp_replace, lpad, lit, when, round, format_number, format_string, date_add, coalesce, split
from botocore.exceptions import NoCredentialsError
from pyspark.sql.window import Window
from pyspark.sql import SparkSession
from pyspark.dbutils import DBUtils
from io import BytesIO
import pandas as pd
import boto3
import json
import os

In [0]:
spark = SparkSession.builder.getOrCreate()
dbutils = DBUtils(spark)

# Function to get AWS credentials from Databricks secrets
def get_aws_credentials(secrets_scope, key_name):
    try:
        # Databricks secrets configuration
   
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        if aws_secret_key is not None:
            print("AWS secret key retrieved successfully.")
        else:
            print("AWS secret key is None. Please check Databricks secrets configuration.")
 
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None

# Function to get AWS credentials from Databricks secrets
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration
 
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
   
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

# Function to connect to AWS and access SSM parameter
def connect_to_aws_and_access_ssm(ssm_key, ssm_client, aws_secret_key):
 
    if aws_secret_key is None:
        print("AWS secret key not available.")
        return
 
    # Access SSM parameter /cfn/a360/reports_output_path
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )
 
    return response['Parameter']['Value']

# Function to get the environment prefix
def get_env_prefix():
    secrets_scope = "aws-databricks"
    aws_keys = get_scope_key_names(secrets_scope)
    aws_key = aws_keys[0] if aws_keys else None
    # Get AWS secret key from Databricks secrets
    aws_secret_key = get_aws_credentials(secrets_scope,aws_key)
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)
 
    S3_client = boto3.client('s3', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)  

    env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix', ssm_client, aws_secret_key)
    return env_prefix

## Get the needed Perimeter and Parameters

In [0]:
dbutils.widgets.text(
    name = "non_real_estate_portfolio_name",
    defaultValue = "",
    label = "NonRealEstate Portfolio Name"
)

In [0]:
datatape_type = dbutils.widgets.get("datatape_type")

# Define allowed values
allowed_values = {"Classic", "Holiday", "Neptune"}

# Check validity
if datatape_type not in allowed_values:
    raise ValueError(f"Invalid datatape_type: '{datatape_type}'. Must be one of {allowed_values}")

In [0]:
env = get_env_prefix()

In [0]:
try:    
    report_args = dbutils.widgets.get("report_args")
    report_args = report_args.replace("\'", "\"")
except Exception as e:
    raise ValueError("Missing required input: report_args")

# Read JSON
json_data = json.loads(report_args)
 
# Access the values and store them into variables
if "output_file_prefix" not in json_data: raise KeyError("'output_file_prefix' key is missing in the report_args JSON data.")
if "report_output" not in json_data: raise KeyError("'report_output' key is missing in the report_args JSON data.")

output_file_prefix = "".join(json_data['output_file_prefix'])
report_output = json_data['report_output']

In [0]:
run_date = dbutils.widgets.get("run_date")

run_date = spark.sql(f"""                    
                        select 
                            case 
                                when try_cast('{run_date}' as date) is not null then try_cast('{run_date}' as date)
                                else PreviousBusinessDay
                            end as BusinessDate
                        from `{env}-cdh`.reference.ref_calendar
                        where CalendarDate = coalesce(try_cast('{run_date}' as date),cast(getdate() as date))
                    """).first()[0]

In [0]:
# Read the Perimeter into a DataFrame
#perimeter_path = f"`{env}-reportinputfiles`.internal.PoolPerimeter_Incremental"
#Initial_pool = spark.read.table(perimeter_path).toDF("AgreementKey", "IngestionTimestamp")
#
#Initial_pool.createOrReplaceTempView("Initial_pool")
#
#pool = spark.sql("""
#                    SELECT DISTINCT AgreementKey
#                    FROM Initial_pool
#                    WHERE IngestionTimestamp = (SELECT MAX(IngestionTimestamp) FROM Initial_pool)
#                """)
#
#pool.createOrReplaceTempView("pool")

In [0]:
non_real_estate_portfolio_name = dbutils.widgets.get("non_real_estate_portfolio_name")
if non_real_estate_portfolio_name != '':

       nonRealEstateKeys = spark.sql(f"""
                                     
       SELECT PortfolioName, 
              AgreementKey,
              NonRealEstateKey,
              CollateralStatus,
              BusinessDate
       FROM `{env}-cdh`.primary.collateral
       WHERE BusinessDate = '{run_date}'
       AND PortfolioName = '{non_real_estate_portfolio_name}'
       AND NonRealEstateKey IS NOT NULL

       """)
       nonRealEstateKeys.createOrReplaceTempView("nonRealEstateKeys")

       agreementsWithNonRealEstateKeys = spark.sql("""
       SELECT DISTINCT AgreementKey FROM nonRealEstateKeys
       """)
       agreementsWithNonRealEstateKeys.createOrReplaceTempView("agreementsWithNonRealEstateKeys")

       nonRealEstateStats = spark.sql("""
                                      
       SELECT BusinessDate, 
              PortfolioName,
              COUNT(*) AS CountRows,
              COUNT(DISTINCT AgreementKey) AS DistinctAgreementKeys,
              COUNT(DISTINCT NonRealEstateKey) AS DistinctNonRealEstateKeys,
              COUNT(DISTINCT CollateralStatus) AS ActiveIncativeStatus
       FROM nonRealEstateKeys
       GROUP BY BusinessDate, PortfolioName  

       """)
       nonRealEstateStats.createOrReplaceTempView("nonRealEstateStats")

       ################################################################

       spark.sql(f"TRUNCATE TABLE `{env}-cdh`.reference.pool")

       spark.sql(f"""
                 INSERT INTO `{env}-cdh`.reference.pool (AgreementKey)
                 SELECT AgreementKey
                 FROM agreementsWithNonRealEstateKeys
                 """)
       
       display(nonRealEstateStats)

In [0]:
pool_reference = spark.sql(f"""SELECT DISTINCT AgreementKey FROM  `{env}-cdh`.reference.pool""")
pool_reference.createOrReplaceTempView("pool_reference")

## Read Source Tables

In [0]:
dparty = spark.sql(f"""
                        select distinct
                                  CDI
                                , PartyKey
                                , GroupCode
                                , GroupName
                                , EconomicActivityCode
                                , LegalNameGR
                                , TaxRegistrationNumber
                                , PartyType
                                , LegalForm
                                , LastNameGR
                                , PartyKeyThesis
                        from `{env}-cdh`.derivative.dparty
                        where BusinessDate = '{run_date}'
                    """)

dparty.createOrReplaceTempView("dparty")
#display(dparty)

In [0]:
rhodium_deleted_agreements = spark.sql(f"""
     SELECT * FROM `p-cdh`.reference.rhodium_deleted_agreements                                  
""")
rhodium_deleted_agreements.createOrReplaceTempView("rhodium_deleted_agreements")

In [0]:
party = spark.sql(f"""
                        select distinct
                                  CDI
                                , RepresentedByPartyKey
                                , PartyKey
                                , SourceSystem
                                , EconomicActivityCode
                                , LegalNameGR
                                , TaxRegistrationNumber
                                , GroupCode
                                , PartyType
                                , LegalForm
                        from `{env}-cdh`.primary.party
                        where BusinessDate = '{run_date}'
                    """)

party.createOrReplaceTempView("party")

In [0]:
nacecode = spark.sql(f"""
                        select distinct
                                 cast( NaceCodeId as int)
                                , Nace1lvlDescGR
                        from `{env}-cdh`.reference.nacecode
                    """)

nacecode.createOrReplaceTempView("nacecode")
#display(nacecode)

In [0]:
Initial_exposure = spark.sql(f"""
                                select distinct
                                          AgreementKey
                                        , ExposureKey
                                        , PartyRelationshipType
                                        , CDI
                                        , ProductKey
                                        , SourceSystem
                                        , isactiveflag
                                from `{env}-cdh`.primary.exposure
                                where BusinessDate = '{run_date}' 
                                and SourceSystem in ('profits', 'qca')
                            """)

Initial_exposure.createOrReplaceTempView("Initial_exposure")

In [0]:
exposure = spark.sql(f"""
                        select  AgreementKey
                              , ExposureKey
                              , PartyRelationshipType
                              , CDI
                              , '{run_date}' as RunDate
                        from Initial_exposure
                        where SourceSystem = 'profits'
                        and isactiveflag = true 
                        union
                        select  AgreementKey
                              , ExposureKey
                              , PartyRelationshipType
                              , CDI
                              , '{run_date}' as RunDate
                        from Initial_exposure
                        where SourceSystem='profits'
                        and isactiveflag = false
                    """)

exposure.createOrReplaceTempView("exposure")

In [0]:
dexposure = spark.sql(f"""
                            select distinct
                                     ExposureKey 
                                   , AgreementKey
                                   , CDI
                                   , PartyRelationshipType
                                   , Currency
                                   , ExposureStatus
                                   , AssetSegment
                                   , Mechanism
                                   , Opendate
                                   , OutstandingTotalBalance
                                   , ExposureStatusCode
                                   , OutstandingTotalBalance
                                   , DueAccrued
                                   , ProductKey
                                   , DaysPastDueWithMateriality
                                   , LastRestructuringType
                                   , LastRestructuringDate
                                   , DenouncedDate
                                   , LPS
                                   , Accrued
                                   , BaseRate
                                   , SpreadRate
                                   , N128Rate
                                   , PenaltyRate
                                   , BaseRate
                                   , SpreadRate
                                   , N128Rate
                                   , PenaltyRate
                                   , Maturitydate
                                   , Interest  
                                   , Commission
                                   , Principal
                                   , Expense
                                   , PenaltyInterest
                                   , UnrealizedInterest
                                   , DuePrincipal
                                   , DueInterest
                                   , DueExpense
                                   , DueCommission
                                   , Accrued
                                   , DueAccrued
                                   , DueUnrealizedInterest
                                   , PenaltyUnrealizedInterest
                                   , InterestValidityDate
                                   , InterestDuration
                                   , InterestRateType
                                   , InterestCode
                                   , InterestDescription
                                   , BaseCode
                                   , BaseDescription
                                   , InterestFrequency
                                   , cast(InterestCalcStopFlag as boolean) as InterestCalcStopFlag
                                   , InterestFixedDate
                            from `{env}-cdh`.derivative.dexposure
                            where BusinessDate = '{run_date}'
                        """)

dexposure.createOrReplaceTempView("dexposure")

In [0]:
Initial_product = spark.sql(f"""
                                select distinct
                                          ProductKey
                                        , ProductDescription
                                        , SourceSystem
                                from `{env}-cdh`.primary.product
                                where BusinessDate = '{run_date}' 
                                and SourceSystem in ('profits', 'qca')
                            """)

Initial_product.createOrReplaceTempView("Initial_product")

In [0]:
product = spark.sql(f"""
                        select distinct
                                  ProductKey
                                , ProductDescription
                        from Initial_product
                        where SourceSystem = 'profits'
                    """)

product.createOrReplaceTempView("product")

In [0]:
subproduct = spark.sql(f"""
                            select distinct
                                      ExposureKey
                                    , ex.ProductKey SubProductKey
                                    , p.ProductDescription SubProductDescription
                            from Initial_exposure ex
                            inner join Initial_product p
                            on ex.ProductKey = p.ProductKey
                            where ex.sourcesystem = 'qca'
                            and p.sourcesystem = 'qca'
                        """)

subproduct.createOrReplaceTempView("subproduct")
#display(subproduct)

In [0]:
dagreement = spark.sql(f"""
                            select distinct
                                      AgreementKey
                                    , NumberOfRestructurings
                                    , IssueDate
                            from `{env}-cdh`.derivative.dagreement
                            where BusinessDate = '{run_date}'
                        """)

dagreement.createOrReplaceTempView("dagreement")

In [0]:
dunderlaw = spark.sql(f"""
                        select distinct
                                    CDI
                                  , UnderLawType
                                  , LawStatus
                                  , ApplicationStatus
                        from `{env}-cdh`.derivative.dunderlaw
                        where BusinessDate = '{run_date}' 
                      """)

dunderlaw.createOrReplaceTempView("dunderlaw")

In [0]:
contactdetails = spark.sql(f"""
                                select distinct 
                                          PartyKey
                                        , CommunicationFlag
                                        , City
                                        , BusinessDate
                                from `{env}-cdh`.primary.contactdetails
                                where BusinessDate = '{run_date}'
                            """)

contactdetails.createOrReplaceTempView("contactdetails")
#display(contactdetails)

In [0]:
dcollateral = spark.sql(f"""
                            select distinct
                                      AgreementKey
                                    , CollateralCode
                                    , CollateralStatus
                                    , StatusDate
                                    , Category
                                    , SubCategory
                                    , CollateralAmount
                                    , Currency
                                    , MortgageType
                                    , Comment
                                    , RealEstateStatus
                                    , RealEstateKey
                                    , LandRegistry
                                    , CourtNumber
                                    , RealEstateCollateralKey
                                    , OwnershipKey
                                    , InitialLegalOrder
                                    , JudgementNumber
                                    , JudgementDate
                                    , TranscriptNumber
                                    , TranscriptSheet
                                    , TranscriptVolume
                                    , Comment
                                    , NonRealEstateKey
                                    , CollateralStatusCode
                            from `{env}-cdh`.derivative.dcollateral
                            where BusinessDate = '{run_date}'
                        """)

dcollateral.createOrReplaceTempView("dcollateral")

In [0]:
payment = spark.sql(f"""
                        select distinct
                                  ExposureKey
                                , PaymentActualDate
                                , PaymentUniqueNumber
                                , Currency
                                , PaymentType
                                , TransactionReason
                                , PaymentAmount
                                , PrincipalAmount
                                , CommissionAmount
                                , InterestAmount
                                , ExpensesAmount
                        from `{env}-cdh`.primary.payment
                        where BusinessDate = '{run_date}'
                    """)

payment.createOrReplaceTempView("payment")

In [0]:
dpaymentschedule = spark.sql(f"""
                                select distinct
                                          ExposureKey
                                        , InstallmentDate
                                        , InstallmentAmount
                                        , InstallmentNumber
                                        , InstallmentType
                                        , InstallmentStatus
                                from `{env}-cdh`.derivative.dpaymentschedule
                                where BusinessDate = '{run_date}'
                            """)

dpaymentschedule.createOrReplaceTempView("dpaymentschedule")

In [0]:
realestatebuilding = spark.sql(f"""
                                    select distinct
                                                RealEstateCepalKey
                                              , RealEstateKey
                                              , RealEstateDateEntered
                                              , RealEstateBuildingFloorRawValue
                                              , RealEstateCategory
                                              , RealEstateSubCategory
                                              , RealEstateUsageCodeType
                                              , RealEstateDescription
                                              , RealEstateArea
                                              , RealEstateCity
                                              , RealEstateAddress1
                                              , RealEstatePostalCode
                                              , RealEstateGeolocation
                                              , RealEstateConstructionYear
                                              , RealEstateBuildingNumber
                                              , RealEstateLandArea
                                              , RealEstateNetArea
                                              , RealEstateBuildingStorageArea
                                              , RealEstateBuildingParkingArea
                                              , RealEstateMarketValue
                                              , RealEstateValueDate
                                              , RealEstateEvaluationDate
                                              , RealEstateValueType
                                              , RealEstateEvaluationType
                                              , RealEstateMarketabilityCode
                                              , RealEstateStatus
                                              , RealEstateDistrict
                                              , RealEstateMunicipality
                                              , RealEstateDivision
                                    from `{env}-cdh`.primary.realestatebuilding
                                    where BusinessDate = '{run_date}'
                                    and sourcesystem = 'thesis'
                                """)

realestatebuilding.createOrReplaceTempView("realestatebuilding")

In [0]:
realestateinfo = spark.sql(f"""
                                select distinct
                                          RealEstateKey
                                        , OwnershipKey
                                        , OwnershipType
                                        , OwnershipPercentage
                                        , OwnerPartyKey
                                        , RELienCreditorKey
                                        , RELienIsActiveFlag
                                        , RELienKey
                                        , RELienInitialOrder
                                        , RELienMortgageAmount
                                from `{env}-cdh`.primary.realestateinfo
                                where BusinessDate = '{run_date}'
                                and OwnershipStatusCode = 'Active'
                            """)

realestateinfo.createOrReplaceTempView("realestateinfo")

In [0]:
nonrealestate = spark.sql(f"""
                                select distinct
                                          NonRealEstateKey
                                        , NonRealEstateDescription
                                        , NonRealEstateAppliedDate
                                        , CashAccountNumber
                                        , CashAccountBank
                                        , NonRealEstateComment
                                        , OwnerPartyKey
                                from `{env}-cdh`.primary.nonrealestate
                                where BusinessDate = '{run_date}'
                            """)

nonrealestate.createOrReplaceTempView("nonrealestate")

In [0]:
legalaction = spark.sql(f"""
                            select distinct
                                      ExposureKey
                                    , LegalActionType
                                    , LegalActionDate
                            from `{env}-cdh`.primary.legalaction
                            where BusinessDate = '{run_date}'
                        """)

legalaction.createOrReplaceTempView("legalaction")

In [0]:
landregistry = spark.sql(f"""
                            select distinct
                                      id
                                    , Description
                            from `{env}-cdh`.reference.thesis_land_registry_mapping
                        """)

landregistry.createOrReplaceTempView("landregistry")

In [0]:
countycourt = spark.sql(f"""
                            select distinct
                                      id
                                    , Description
                            from `{env}-cdh`.reference.thesis_county_court_mapping
                        """)

countycourt.createOrReplaceTempView("countycourt")

In [0]:
latest_contact = spark.sql(f"""
                                SELECT PartyKey, City, BusinessDate
                                FROM (
                                    SELECT PartyKey
                                        , City
                                        , BusinessDate
                                        , ROW_NUMBER() OVER (PARTITION BY PartyKey ORDER BY BusinessDate DESC) AS rn
                                    FROM contactdetails
                                    WHERE CommunicationFlag = '1'
                                ) sub
                                WHERE rn = 1
                            """)

latest_contact.createOrReplaceTempView("latest_contact")
#display(latest_contact)

In [0]:
latest_legalaction = spark.sql(f"""
                                    SELECT ExposureKey, LegalActionType, LegalActionDate
                                    FROM (
                                        SELECT *,
                                        ROW_NUMBER() OVER (PARTITION BY ExposureKey ORDER BY LegalActionDate DESC) AS rn
                                        FROM legalaction
                                    ) sub
                                    WHERE rn = 1
                                """)

latest_legalaction.createOrReplaceTempView("latest_legalaction")

In [0]:
MasterSlave = spark.sql(f"""
                            select EntityKey MasterAccountNumber
                                    , RelatedEntityKey SlaveAccountNumber
                                    , EntityRelationshipSubType RelationType
                            from `{env}-cdh`.derivativeactive.dentityrelationships
                            where businessdate = '{run_date}'
                        """)

MasterSlave.createOrReplaceTempView("MasterSlave")

In [0]:
prcollateraldet = spark.sql(f"""
                                SELECT DISTINCT
                                          collateral_id
                                        , item_descr
                                FROM `{env}-cdh`.landed.thesis_prcollateraldet
                                WHERE BusinessDate = '{run_date}'
                            """)

prcollateraldet.createOrReplaceTempView("prcollateraldet")

In [0]:
prcollateral = spark.sql(f"""
                            SELECT DISTINCT
                                      id
                                    , reference_key
                            FROM `{env}-cdh`.landed.thesis_prcollateral
                            WHERE BusinessDate = '{run_date}'
                        """)

prcollateral.createOrReplaceTempView("prcollateral")

In [0]:
prlinked_properties = spark.sql(f"""
                                    SELECT DISTINCT
                                              collateral_ids
                                            , auction_code
                                            , bidder_id
                                            , collateral_id
                                            , property_id
                                    FROM `{env}-cdh`.landed.thesis_prlinked_properties
                                    WHERE BusinessDate = '{run_date}'
                                """)

prlinked_properties.createOrReplaceTempView("prlinked_properties")

In [0]:
prproperty = spark.sql(f"""
                            SELECT DISTINCT
                                      cepal_key
                                    , id
                            FROM `{env}-cdh`.landed.thesis_prproperty
                            WHERE BusinessDate = '{run_date}'
                        """)

prproperty.createOrReplaceTempView("prproperty")

In [0]:
prperson = spark.sql(f"""
                        SELECT DISTINCT
                                  id
                                , last_name
                        FROM `{env}-cdh`.landed.thesis_prperson
                        WHERE BusinessDate = '{run_date}'
                    """)

prperson.createOrReplaceTempView("prperson")

In [0]:
profits_account = spark.sql(f"""
                                SELECT DISTINCT
                                          ACCOUNT_NUMBER
                                        , PRFT_SYSTEM
                                        , LNS_OPEN_UNIT
                                        , LNS_SN
                                        , LNS_TYPE
                                FROM `{env}-cdh`.landed.profits_profits_account
                                WHERE BusinessDate = '{run_date}'
                            """)

profits_account.createOrReplaceTempView("profits_account")

In [0]:
datetape_hie = spark.sql(f"""
                            SELECT DISTINCT
                                      collateral_id
                                    , last_name
                            FROM `{env}-cdh`.reference.datetape_hie
                        """)

datetape_hie.createOrReplaceTempView("datetape_hie")

In [0]:
itstaging_loanrequest = spark.sql(f"""
                                        SELECT DISTINCT
                                                  FK_LOAN_ACCOUNTACC
                                                , FK_LOAN_ACCOUNTFK
                                                , FK0LOAN_ACCOUNTACC
                                                , REQUEST_TYPE
                                                , REQUEST_LOAN_STS
                                                , REQUEST_STS
                                                , RQ_EXPIRE_DT
                                                , RQ_CAPITAL_BAL
                                                , RQ_RL_NRM_INT_BAL 
                                                , RQ_RL_PNL_INT_BAL 
                                                , RQ_EXP_BAL
                                                , RQ_COM_BAL
                                                , RQ_URL_NRM_INT_BAL
                                                , OV_ACCRUAL_AMN
                                        FROM `p-cdh`.landed.profits_loanrequest
                                        WHERE BusinessDate = '{run_date}'
                                    """)

itstaging_loanrequest.createOrReplaceTempView("itstaging_loanrequest")

## Prepare Tables

### Loan

In [0]:
Loan = spark.sql(f"""
                    SELECT DISTINCT
                              '{run_date}' AS Cutoff_Date
                            , dexp.ExposureKey AS Loan_ID
                            , pool.AgreementKey AS Contract_ID
                            , dpa.CDI AS Borrower_ID
                            , dpa.GroupCode AS Group_ID
                            , dexp.Currency AS Currency
                            , dexp.ExposureStatus AS LoanStatus
                            , dexp.AssetSegment AS Asset_Class
                            , dexp.Mechanism AS Account_Amortisation_Type
                            , CASE 
                                WHEN TO_DATE(dag.IssueDate, 'yyyy-MM-dd HH:mm:ss') < DATE('1900-01-01') THEN NULL
                                ELSE TO_DATE(dag.IssueDate, 'yyyy-MM-dd HH:mm:ss')
                              END AS Contract_Signing_Date
                            , dexp.Opendate AS Account_Opening_Date
                            , try_cast(dexp.OutstandingTotalBalance AS DOUBLE) AS Total_Exposure_EUR
                            , CASE 
                                WHEN dexp.ExposureStatusCode <> '4' THEN try_cast(dexp.OutstandingTotalBalance AS DOUBLE) 
                                ELSE 0 
                              END AS Total_ON_Balance_EUR
                            , CASE 
                                WHEN dexp.ExposureStatusCode <> '4' THEN 0
                                ELSE try_cast(dexp.OutstandingTotalBalance AS DOUBLE)
                              END AS Total_OFF_Balance_EUR
                            , round(try_cast(dexp.DueAccrued AS DOUBLE) + try_cast(dexp.Accrued AS DOUBLE), 2) AS Accruals
                            , CASE 
                                WHEN split(dexp.ProductKey, ';')[1] in ('1220','1225','1226','1228','3006','3107','3403','7734','7738','7817') THEN "Yes" 
                                ELSE "No" 
                              END AS Syndicated_Flag
                            , cast(NULL as int) AS `Syndication_Participation%`
                            , dexp.DaysPastDueWithMateriality AS Days_Past_Due
                            , cast(NULL as int) AS Number_of_Forbearances
                            , cast(NULL as int) AS Forbearance_Date
                            , dexp.LastRestructuringType AS Restructuring_Type
                            , CASE 
                                WHEN dexp.LastRestructuringDate <= '1900-01-01' THEN NULL 
                                ELSE dexp.LastRestructuringDate 
                              END AS Last_Restructuring_Date
                            , dag.NumberOfRestructurings AS Number_of_Restructurings
                            , CASE 
                                WHEN dexp.ExposureStatusCode IN ('3', '4') THEN 'Yes' 
                                ELSE 'No' 
                              END AS Denouncement
                            , dexp.DenouncedDate AS Denouncement_Date
                            , CASE 
                                WHEN law.CDI IS NULL THEN "No" 
                                ELSE "Yes" 
                              END AS L_3869_Protection_Flag
                            , law.LawStatus AS L_3869_Classification
                            , la.LegalActionType AS Last_Legal_Action
                            , la.LegalActionDate AS Last_Legal_Action_Date
                            , dexp.LPS AS Legal_Process_Status
                            , round(try_cast(dexp.BaseRate AS DOUBLE), 2) AS Base_Interest
                            , round(try_cast(dexp.SpreadRate AS DOUBLE), 2) AS Spread
                            , round(try_cast(dexp.N128Rate AS DOUBLE), 2) AS N128
                            , round(try_cast(dexp.PenaltyRate AS DOUBLE), 2) AS Penalty
                            , round( 
                                    ( try_cast(dexp.BaseRate AS DOUBLE) 
                                    + try_cast(dexp.SpreadRate AS DOUBLE) 
                                    + try_cast(dexp.N128Rate AS DOUBLE) 
                                    + CASE 
                                          WHEN dexp.ExposureStatusCode = '1' THEN 0 
                                          ELSE try_cast(dexp.PenaltyRate AS DOUBLE) 
                                      END)
                                    , 2) AS Interest_Rate
                            , replace(dexp.ProductKey, ';', '-') AS ProductKey
                            , prod.ProductDescription
                    FROM  pool_reference pool
                    INNER JOIN dexposure dexp
                        ON dexp.AgreementKey = pool.AgreementKey
                    LEFT JOIN dparty dpa
                        ON dexp.CDI = dpa.CDI
                    LEFT JOIN dagreement dag
                        ON dag.AgreementKey = dexp.AgreementKey
                    LEFT JOIN dunderlaw law
                        ON law.CDI = dexp.CDI
                        AND (ApplicationStatus like '%Accepted%' or ApplicationStatus like '%Approved%' or ApplicationStatus like '%Pending%')
                        AND law.UnderLawType IN ("N3869")
                    LEFT JOIN product prod
                        ON prod.ProductKey = dexp.ProductKey
                    LEFT JOIN latest_legalaction la
                        ON la.ExposureKey = SUBSTRING(dexp.ExposureKey, 4, 50)
                    WHERE NOT EXISTS (
                        SELECT 1
                        FROM rhodium_deleted_agreements rda
                        WHERE rda.AgreementKey = pool.AgreementKey
                          AND rda.ExposureKey = dexp.ExposureKey
                    )
                """)

Loan.createOrReplaceTempView("Loan")
#display(Loan)

### Relations

In [0]:
Relations = spark.sql(f"""
                            SELECT DISTINCT
                                  exp.AgreementKey AS ContractId
                                , exp.ExposureKey AS Loan_ID
                                , exp.CDI AS BorrowerId
                                , exp.PartyRelationshipType AS Relation
                                , rep.CDI AS Representative
                            FROM pool_reference  pool
                            INNER JOIN exposure exp
                                ON exp.AgreementKey = pool.AgreementKey
                            INNER JOIN party p
                                ON exp.CDI = p.CDI
                                and p.SourceSystem = 'profits'
                            LEFT JOIN party rep
                                ON p.RepresentedByPartyKey = rep.PartyKey
                                and p.SourceSystem = rep.sourceSystem
                            WHERE NOT EXISTS (
                                SELECT 1
                                FROM rhodium_deleted_agreements rda
                                WHERE rda.AgreementKey = pool.AgreementKey
                                  AND rda.ExposureKey = exp.ExposureKey
                            )
                        """)

Relations.createOrReplaceTempView("Relations")
#display(Relations)

### Representative

In [0]:
Representative = spark.sql(f"""
                                SELECT DISTINCT
                                          rep.CDI AS Representative_ID
                                       , rep.LegalNameGR AS Representative_Name
                                      , rep.TaxRegistrationNumber AS TaxNumber
                                      , rep.GroupCode AS Group_ID
                                      , rep.PartyType AS Legal_Type
                                      , rep.LegalForm AS Legal_Form
                                      , nc.Nace1lvlDescGR AS Representative_Industry
                                     , con.City AS Region
                                FROM  pool_reference pool
                                INNER JOIN exposure exp
                                    ON exp.AgreementKey = pool.AgreementKey 
                                INNER JOIN party p 
                                    ON exp.CDI = p.CDI
                                INNER JOIN party rep 
                                    ON p.RepresentedByPartyKey = rep.PartyKey and rep.LegalNameGR is not null
                                LEFT JOIN nacecode nc 
                                    ON rep.EconomicActivityCode  = nc.NaceCodeId
                                LEFT JOIN latest_contact con 
                                    ON rep.PartyKey = con.PartyKey
                            """)

Representative.createOrReplaceTempView("Representative")
#display(Representative)

### Borrower

In [0]:
Borrower = spark.sql(f"""
                        SELECT DISTINCT
                                  dp.CDI AS Borrower_ID
                                , dp.LegalNameGR AS Borrower_Name
                                , dp.TaxRegistrationNumber AS TaxNumber
                                , dp.GroupCode AS Group_ID
                                , dp.PartyType AS Legal_Type
                                , dp.LegalForm AS Legal_Form
                                , cast(NULL as int) AS Borrower_Segment
                                , nc.Nace1lvlDescGR AS Borrower_Industry
                                , cast(NULL as int) AS Country_of_Establishment
                                , con.City AS Region
                                , cast(NULL as int) AS Operating_Flag
                                , cast(NULL as int) AS Cooperative_Flag
                                , cast(NULL as int) AS Bankruptcy_Insolvency_Process_Status
                        FROM  pool_reference  pool
                        INNER JOIN exposure exp
                            ON exp.AgreementKey = pool.AgreementKey
                        INNER JOIN dparty dp 
                            ON exp.CDI = dp.CDI
                        LEFT JOIN nacecode nc 
                            ON dp.EconomicActivityCode = nc.NaceCodeId
                        LEFT JOIN latest_contact con 
                            ON dp.PartyKey = con.PartyKey
                        WHERE NOT EXISTS (
                            SELECT 1
                            FROM rhodium_deleted_agreements rda
                            WHERE rda.AgreementKey = pool.AgreementKey
                              AND rda.ExposureKey = exp.ExposureKey
                        )
                    """)

Borrower.createOrReplaceTempView("Borrower")
#display(Borrower)

### Collateral

In [0]:
Collateral = spark.sql(f"""
                          SELECT DISTINCT 
                                dcol.AgreementKey AS contractId
                              , dcol.CollateralCode AS Collateral_code
                              , dcol.CollateralStatus AS Collateral_Status
                              , CASE 
                                  WHEN dcol.StatusDate <= '1900-01-01' THEN NULL
                                  ELSE dcol.StatusDate
                                END AS CollateralStatusChangeDate
                              , dcol.Category AS Collateral_Category
                              , dcol.SubCategory AS SubCategory
                              , dcol.CollateralAmount AS Collateral_Amount
                              , dcol.Currency AS Currency
                              , dcol.MortgageType AS Mortgage_Type
                              , dcol.Comment AS Comments
                              , 'NO' AS existInOffPerimeterContracts
                          FROM  pool_reference  pool
                          INNER JOIN dcollateral dcol
                              ON dcol.AgreementKey = pool.AgreementKey
                          AND dcol.CollateralStatus = "Active"
                          WHERE NOT EXISTS (
                              SELECT 1
                              FROM rhodium_deleted_agreements rda
                              WHERE rda.AgreementKey = pool.AgreementKey
                          )
                        """)

Collateral.createOrReplaceTempView("Collateral")
#display(Collateral)

### CollateralRE

In [0]:
CollateralRE = spark.sql("""
                            SELECT 
                                x.*,
                                CASE 
                                    WHEN COUNT(x.Property_ID) OVER (PARTITION BY Collateral_code) > 1 THEN 'YES'
                                    ELSE 'NO'
                                END AS Common_Prenotation_Flag
                            FROM (
                                SELECT DISTINCT
                                      dcol.CollateralCode AS Collateral_code
                                    , dcol.RealEstateCollateralKey AS CollateralRE_Id
                                    , dcol.RealEstateStatus AS CollateralRE_Status
                                    , dcol.OwnershipKey AS `Ownership ID`
                                    , building.RealEstateCepalKey AS Property_ID
                                    , dcol.InitialLegalOrder AS `Lien_Order`
                                    , dcol.MortgageType AS `Prenotation_Type`
                                    , building.RealEstateDateEntered AS `Registration_Date`
                                    , dcol.JudgementNumber AS `Αρ Δικαστικής Απόφασης`
                                    , CASE 
                                        WHEN dcol.JudgementDate <= '1900-01-01' THEN NULL
                                        ELSE dcol.JudgementDate
                                      END AS `Ημ/νια Δικαστικής Απόφασης`
                                    , cc.Description AS `Ειρηνοδικείο/Πρωτοδικείο`
                                    , lrm.Description AS `Υποθηκοφυλακείο`
                                    , dcol.TranscriptNumber AS `Αριθμός Μεττεγραφής`
                                    , dcol.TranscriptSheet AS `Φύλλο`
                                    , dcol.TranscriptVolume AS `Τόμος`
                                    , dcol.Comment AS `Σχόλια`
                                FROM  pool_reference  pool
                                INNER JOIN dcollateral dcol
                                    ON dcol.AgreementKey = pool.AgreementKey
                                    AND dcol.RealEstateStatus = "Active"
                                LEFT JOIN realestatebuilding building
                                    ON dcol.RealEstateKey = building.RealEstateKey
                                LEFT JOIN landregistry lrm
                                    ON lrm.id = dcol.LandRegistry
                                LEFT JOIN countycourt cc
                                    ON cc.id = dcol.CourtNumber
                                WHERE NOT EXISTS (
                                    SELECT 1
                                    FROM rhodium_deleted_agreements rda
                                    WHERE rda.AgreementKey = pool.AgreementKey
                                )    
                            ) x
                        """)

CollateralRE.createOrReplaceTempView("CollateralRE")
display(CollateralRE)

### RE

In [0]:
modified_realestatebuilding = spark.sql(f"""
                                            SELECT
                                                coalesce(pb.RealEstateCepalKey, store.RealEstateCepalKey, park.RealEstateCepalKey) AS RealEstateCepalKey,
                                                dcol.RealEstateKey,
                                                coalesce(pb.RealEstateCategory, store.RealEstateCategory, park.RealEstateCategory) AS RealEstateCategory,
                                                coalesce(pb.RealEstateSubCategory, store.RealEstateSubCategory, park.RealEstateSubCategory) AS RealEstateSubCategory,
                                                coalesce(pb.RealEstateUsageCodeType, store.RealEstateUsageCodeType, park.RealEstateUsageCodeType) AS RealEstateUsageCodeType,
                                                coalesce(pb.RealEstateDescription, store.RealEstateDescription, park.RealEstateDescription)  AS RealEstateDescription,
                                                coalesce(pb.RealEstateArea, store.RealEstateArea, park.RealEstateArea) AS RealEstateArea,
                                                coalesce(pb.RealEstateCity, store.RealEstateCity, park.RealEstateCity) AS RealEstateCity,
                                                coalesce(pb.RealEstateAddress1, store.RealEstateAddress1, park.RealEstateAddress1) AS RealEstateAddress1,
                                                coalesce(pb.RealEstatePostalCode, store.RealEstatePostalCode, park.RealEstatePostalCode) AS RealEstatePostalCode,
                                                coalesce(pb.RealEstateGeolocation, store.RealEstateGeolocation, park.RealEstateGeolocation) AS RealEstateGeolocation,
                                                coalesce(pb.RealEstateConstructionYear, store.RealEstateConstructionYear, park.RealEstateConstructionYear) AS RealEstateConstructionYear,
                                                coalesce(pb.RealEstateBuildingNumber, store.RealEstateBuildingNumber, park.RealEstateBuildingNumber) AS RealEstateBuildingNumber,
                                                coalesce(pb.RealEstateLandArea, store.RealEstateLandArea, park.RealEstateLandArea) AS RealEstateLandArea,
                                                coalesce(pb.RealEstateNetArea, store.RealEstateNetArea, park.RealEstateNetArea) AS RealEstateNetArea,
                                                store.RealEstateBuildingStorageArea AS RealEstateBuildingStorageArea,
                                                pb.RealEstateBuildingFloorRawValue AS building_RealEstateBuildingFloorRawValue,
                                                store.RealEstateBuildingFloorRawValue AS store_RealEstateBuildingFloorRawValue,
                                                park.RealEstateBuildingFloorRawValue AS park_RealEstateBuildingFloorRawValue,
                                                park.RealEstateBuildingParkingArea  AS RealEstateBuildingParkingArea,
                                                coalesce(pb.RealEstateMarketValue, store.RealEstateMarketValue, park.RealEstateMarketValue) AS RealEstateMarketValue,
                                                coalesce(pb.RealEstateValueDate, store.RealEstateValueDate, park.RealEstateValueDate) AS RealEstateValueDate,
                                                coalesce(pb.RealEstateEvaluationDate, store.RealEstateEvaluationDate, park.RealEstateEvaluationDate) AS RealEstateEvaluationDate,
                                                coalesce(pb.RealEstateValueType, store.RealEstateValueType, park.RealEstateValueType) AS RealEstateValueType,
                                                coalesce(pb.RealEstateEvaluationType, store.RealEstateEvaluationType, park.RealEstateEvaluationType) AS RealEstateEvaluationType,
                                                coalesce(pb.RealEstateMarketabilityCode, store.RealEstateMarketabilityCode, park.RealEstateMarketabilityCode) AS RealEstateMarketabilityCode
                                            FROM dcollateral dcol
                                            left join realestatebuilding pb
                                            on dcol.RealEstateKey = pb.RealEstateKey
                                            and pb.RealEstateBuildingFloorRawValue not in (96,-99)
                                            left join realestatebuilding store
                                            on dcol.RealEstateKey = store.RealEstateKey
                                            and store.RealEstateBuildingFloorRawValue = -99
                                            left join realestatebuilding park
                                            on dcol.RealEstateKey = park.RealEstateKey
                                            and park.RealEstateBuildingFloorRawValue = 96
                                        """)

modified_realestatebuilding.createOrReplaceTempView("modified_realestatebuilding")

In [0]:
RE = spark.sql("""
                    SELECT DISTINCT
                          building.RealEstateCepalKey AS Property_ID
                        , building.RealEstateCategory AS Property_Type
                        , building.RealEstateSubCategory AS Property_Sub_Type
                        , building.RealEstateUsageCodeType AS `Usage Type`
                        , building.RealEstateDescription AS RE_Asset_Description
                        , building.RealEstateArea AS Prefecture
                        , building.RealEstateCity AS Municipality
                        , building.RealEstateAddress1 AS Street_Name
                        , CASE 
                            WHEN building.RealEstatePostalCode IN ('10000', '99999') THEN NULL 
                            ELSE building.RealEstatePostalCode 
                          END AS Postal_Code
                        , building.RealEstateGeolocation AS GPS_Coordinates
                        , building.RealEstateConstructionYear AS Construction_Year
                        , building.RealEstateBuildingNumber AS Building_no
                        , CASE
                            WHEN building.RealEstateLandArea < 1 THEN NULL
                            ELSE building.RealEstateLandArea
                          END AS `Land Area sqm`
                        , building.RealEstateNetArea AS Main_Uses_sqm
                        , mf.Main_Uses_Floor_Number AS Main_Uses_Floor_Number
                        , building.RealEstateBuildingStorageArea AS Storage_Area_sqm
                        , CASE 
                            WHEN building.RealEstateBuildingStorageArea IS NOT NULL AND building.RealEstateBuildingStorageArea <> '0' THEN 
                                CASE 
                                    WHEN building.store_RealEstateBuildingFloorRawValue = -99 THEN 'ΑΓΝΩΣΤΟ' 
                                    ELSE CAST(building.store_RealEstateBuildingFloorRawValue AS STRING) 
                                END
                            ELSE NULL 
                          END AS Storage_Floor_Number
                        , building.RealEstateBuildingParkingArea AS Parking_Space_Area_sqm
                        , CASE 
                            WHEN building.RealEstateBuildingParkingArea IS NOT NULL AND building.RealEstateBuildingParkingArea <> '0' THEN 
                                CASE 
                                    WHEN building.park_RealEstateBuildingFloorRawValue = 96 THEN 'ΘΕΣΗ ΣΤΑΘΜΕΥΣΗΣ' 
                                    ELSE CAST(building.park_RealEstateBuildingFloorRawValue AS STRING) 
                                END 
                            ELSE NULL
                          END AS Parking_Space_Floor_Number
                        , round(building.RealEstateMarketValue, 2) AS Last_Appraisal_Value
                        , COALESCE(building.RealEstateValueDate, building.RealEstateEvaluationDate) AS Last_Appraisal_Date
                        , CASE COALESCE(building.RealEstateValueType, building.RealEstateEvaluationType)
                            WHEN 'A' THEN 'Indexed'
                            WHEN 'D' THEN 'Desktop'
                            WHEN 'E' THEN 'Drive-By'
                            WHEN 'M' THEN 'Certified'
                            ELSE COALESCE(building.RealEstateValueType, building.RealEstateEvaluationType)
                          END AS `Appraisal Type`
                        , building.RealEstateMarketabilityCode AS Marketability_Score
                    FROM  pool_reference  pool
                    INNER JOIN dcollateral dcol
                        ON dcol.AgreementKey = pool.AgreementKey
                        AND dcol.RealEstateStatus = "Active"
                    LEFT JOIN modified_realestatebuilding building
                        ON dcol.RealEstateKey = building.RealEstateKey
                    LEFT JOIN (
                        SELECT DISTINCT
                              floors.RealEstateCepalKey AS PropertyId
                            , CONCAT_WS(', ', COLLECT_SET(floors.RealEstateBuildingFloorRawValue)) AS Main_Uses_Floor_Number
                        FROM realestatebuilding floors
                        WHERE floors.RealEstateBuildingFloorRawValue NOT IN (96, -99)
                        GROUP BY 1
                    ) mf 
                        ON building.RealEstateCepalKey = mf.PropertyId
                    WHERE NOT EXISTS (
                              SELECT 1
                              FROM rhodium_deleted_agreements rda
                              WHERE rda.AgreementKey = pool.AgreementKey
                          )
                """)

RE.createOrReplaceTempView("RE")
#display(RE)

### ReOwnership

In [0]:
REOwnership = spark.sql("""
                            SELECT DISTINCT
                                  reinfo.OwnershipKey AS PropertyOwnershipId
                                , building.RealEstateCepalKey AS Property_ID
                                , dp.CDI AS CDI
                                , dp.TaxRegistrationNumber as VatNumber
                                , dp.LegalNameGR AS PartyName
                                , reinfo.OwnershipType AS RealTitle_Id
                                , reinfo.OwnershipPercentage AS `Ownership_%`
                            FROM  pool_reference pool
                            INNER JOIN dcollateral dcol
                                ON dcol.AgreementKey = pool.AgreementKey
                            INNER JOIN realestateinfo reinfo
                                ON dcol.RealEstateKey = reinfo.RealEstateKey
                            INNER JOIN realestatebuilding building 
                                ON reinfo.RealEstateKey = building.RealEstateKey
                            INNER JOIN dparty dp
                                ON reinfo.OwnerPartyKey = dp.PartyKeyThesis
                            WHERE building.RealEstateStatus IN ('Ενεργό')
                            AND NOT EXISTS (
                              SELECT 1
                              FROM rhodium_deleted_agreements rda
                              WHERE rda.AgreementKey = pool.AgreementKey
                          )
                        """)

REOwnership.createOrReplaceTempView("REOwnership")
#display(REOwnership)

### ThirdPartyLien

In [0]:
ThirdPartyLien = spark.sql("""
                                SELECT DISTINCT
                                      reinfo.RELienKey AS PropertyLienId
                                    , reinfo.OwnershipKey AS PropertyOwnershipId
                                    , building.RealEstateCepalKey AS Property_ID
                                    , reinfo.RELienInitialOrder AS InitialLegalLienOrder
                                    , round(reinfo.RELienMortgageAmount, 2) AS MortgageAmount
                                    , dp.LastNameGR AS PartyName
                                FROM  pool_reference  pool
                                INNER JOIN dcollateral dcol
                                    ON dcol.AgreementKey = pool.AgreementKey
                                INNER JOIN realestateinfo reinfo
                                    ON dcol.RealEstateKey = reinfo.RealEstateKey
                                INNER JOIN realestatebuilding building 
                                    ON reinfo.RealEstateKey = building.RealEstateKey
                                LEFT JOIN dparty dp
                                    ON reinfo.RELienCreditorKey = dp.PartyKeyThesis
                                WHERE reinfo.RELienIsActiveFlag IS TRUE
                                AND NOT EXISTS (
                                    SELECT 1
                                    FROM rhodium_deleted_agreements rda
                                    WHERE rda.AgreementKey = pool.AgreementKey
                                )
                            """)

ThirdPartyLien.createOrReplaceTempView("ThirdPartyLien")
#display(ThirdPartyLien)

### CollateralNonRE

In [0]:
CollateralNonRE = spark.sql("""
                                SELECT DISTINCT
                                    dcol.CollateralCode AS Collateral_code,
                                    dcol.CollateralStatusCode AS CollateralNRE_status,
                                    nre.NonRealEstateDescription AS `Collateral_Type ID`,
                                    dp.PartyKeyThesis AS AgreementPrimaryOwner,
                                    dp.LegalNameGR AS PrimaryName,
                                    dp2.PartyKeyThesis AS `Ενεχόμενος`,
                                    dp2.LegalNameGR AS `Ενεχόμενος Name`,
                                    DATE(nre.NonRealEstateAppliedDate) AS `Ημερομηνία εγγραφής εξασφάλισης`,
                                    nre.CashAccountNumber AS `ΙΒΑΝ Λογ/μου`,
                                    nre.CashAccountBank AS `Τράπεζα Τήρησης Λογαριασμού`,
                                    nre.NonRealEstateComment AS `Σχόλια`
                                FROM  pool_reference pool 
                                INNER JOIN dcollateral dcol
                                    ON pool.AgreementKey = dcol.AgreementKey
                                INNER JOIN nonrealestate nre 
                                    ON dcol.NonRealEstateKey = nre.NonRealEstateKey
                                INNER JOIN dexposure dexp
                                    ON pool.AgreementKey = dexp.AgreementKey
                                INNER JOIN dparty dp
                                    ON dexp.CDI = dp.CDI
                                LEFT JOIN dparty dp2
                                    ON nre.OwnerPartyKey = dp2.PartyKeyThesis
                                WHERE NOT EXISTS (
                                    SELECT 1
                                    FROM rhodium_deleted_agreements rda
                                    WHERE rda.AgreementKey = pool.AgreementKey
                                      AND rda.ExposureKey = dexp.ExposureKey
                                )
                            """)

CollateralNonRE.createOrReplaceTempView("CollateralNonRE")
#display(CollateralNonRE)

### Payments

In [0]:
Payments = spark.sql("""
                        SELECT DISTINCT
                              dexp.AgreementKey AS ContractID
                            , dexp.ExposureKey AS LoanID
                            , dexp.CDI AS BorrowerID
                            , dp.TaxRegistrationNumber AS BorrowerTaxNumber
                            , pay.PaymentActualDate AS PaymentActualDate
                            , CONCAT(
                               REPLACE(SUBSTRING(split(pay.PaymentUniqueNumber, ';')[0], 1, 10), '-', '')
                               , get(split(pay.PaymentUniqueNumber, ';'), 1)
                               , get(split(pay.PaymentUniqueNumber, ';'), 2)
                               , get(split(pay.PaymentUniqueNumber, ';'), 3)
                              ) AS PaymentUniqueNumber
                            , pay.PaymentUniqueNumber initial
                            , pay.Currency AS Currency
                            , pay.PaymentType AS PaymentCategory
                            , pay.TransactionReason AS TransactionReason
                            , pay.PaymentAmount AS PaymentAmount
                            , pay.PrincipalAmount AS PrincipalAmount
                            , pay.CommissionAmount AS CommissionAmount
                            , pay.InterestAmount AS InterestAmount
                            , pay.ExpensesAmount AS ExpensesAmount
                        FROM  pool_reference  pool
                        INNER JOIN dexposure dexp 
                            ON pool.AgreementKey = dexp.AgreementKey
                        INNER JOIN payment pay 
                            ON pay.ExposureKey = dexp.ExposureKey
                        LEFT JOIN dparty dp 
                            ON dexp.CDI = dp.CDI
                        WHERE NOT EXISTS (
                              SELECT 1
                              FROM rhodium_deleted_agreements rda
                              WHERE rda.AgreementKey = pool.AgreementKey
                                AND rda.ExposureKey = dexp.ExposureKey
                          )
                    """)

Payments.createOrReplaceTempView("Payments")
#display(Payments)

### PaymentPlan

In [0]:
PaymentPlan = spark.sql("""
                            SELECT DISTINCT
                                  dexp.ExposureKey AS AccountNumber
                                , DATE(payplan.InstallmentDate) AS InstallmentDate
                                , round(payplan.InstallmentAmount, 2) AS InstallmentAmount
                                , payplan.InstallmentNumber
                                , payplan.InstallmentType
                                , payplan.InstallmentStatus
                            FROM  pool_reference  pool
                            INNER JOIN dexposure dexp
                                ON pool.AgreementKey = dexp.AgreementKey
                            INNER JOIN dpaymentschedule payplan
                                ON payplan.ExposureKey = dexp.ExposureKey
                                and payplan.InstallmentType in ('Installment', 'Interest Request')
                            WHERE NOT EXISTS (
                              SELECT 1
                              FROM rhodium_deleted_agreements rda
                              WHERE rda.AgreementKey = pool.AgreementKey
                                AND rda.ExposureKey = dexp.ExposureKey
                            )
                        """)

PaymentPlan.createOrReplaceTempView("PaymentPlan") 
#display(PaymentPlan)

## Export

### ExportData

In [0]:
if datatape_type  in  ('Classic','Neptune'):
    LoanTape_sheet = spark.sql("""
                                    SELECT *
                                    FROM Loan
                                """)

In [0]:
if datatape_type  in  ('Classic','Neptune'):
    Relations_sheet = spark.sql("""
                                    SELECT *
                                    FROM Relations
                                """)

In [0]:
if datatape_type  in  ('Classic','Neptune'):
    Representative_sheet = spark.sql("""
                                        SELECT *
                                        FROM Representative
                                    """)

In [0]:
if datatape_type  in  ('Classic','Neptune'):
    Borrower_sheet = spark.sql("""
                                    SELECT *
                                    FROM Borrower
                                """)

In [0]:
if datatape_type  in  ('Classic','Neptune'):
    Collateral_sheet = spark.sql("""
                                    SELECT *
                                    FROM Collateral
                                """)

In [0]:
if datatape_type  in  ('Classic','Neptune'):
    CollateralRE_sheet = spark.sql("""
                                    SELECT *
                                    FROM CollateralRE
                                """)

In [0]:
if datatape_type  in  ('Classic','Neptune'):
    RE_sheet = spark.sql("""
                            SELECT *
                            FROM RE
                        """)

In [0]:
if datatape_type  in  ('Classic','Neptune'):
    REOwnership_sheet = spark.sql("""
                                    SELECT DISTINCT
                                          PropertyOwnershipId AS Property_OwnershipId
                                        , Property_ID
                                        , CDI
                                        , VatNumber
                                        , PartyName
                                        , RealTitle_Id
                                        , `Ownership_%`
                                    FROM REOwnership
                                """)

In [0]:
if datatape_type  in  ('Classic','Neptune'):
    ThirdPartyLien_sheet = spark.sql("""
                                        SELECT *
                                        FROM ThirdPartyLien
                                    """)

In [0]:
if datatape_type  in  ('Classic','Neptune'):
    CollateralNonRE_sheet = spark.sql("""
                                        SELECT *
                                        FROM CollateralNonRE
                                    """)

In [0]:
if datatape_type  in  ('Classic','Neptune'):
    Payments_sheet = spark.sql("""
                                    SELECT *
                                    FROM Payments
                                """)

In [0]:
if datatape_type  in  ('Classic','Neptune'):
    PaymentPlan_sheet = spark.sql("""
                                    SELECT *
                                    FROM PaymentPlan
                                """)

### ExportDataButBack

In [0]:
if datatape_type ==  'Holiday':
    LoanTape_sheet = spark.sql("""
                                SELECT DISTINCT
                                          a.Cutoff_Date
                                        , a.Loan_ID
                                        , a.Contract_ID
                                        , a.Borrower_ID
                                        , a.Group_ID
                                        , dpa.GroupName 
                                        , a.Currency
                                        , a.LoanStatus
                                        , a.Asset_Class
                                        , a.Account_Amortisation_Type
                                        , a.Contract_Signing_Date
                                        , a.Account_Opening_Date
                                        , a.Total_Exposure_EUR
                                        , a.Total_ON_Balance_EUR
                                        , a.Total_OFF_Balance_EUR
                                        , a.Accruals
                                        , a.Syndicated_Flag
                                        , a.`Syndication_Participation%`
                                        , a.Days_Past_Due
                                        , a.Number_of_Forbearances
                                        , a.Forbearance_Date
                                        , a.Restructuring_Type
                                        , a.Last_Restructuring_Date
                                        , a.Number_of_Restructurings
                                        , a.Denouncement
                                        , a.Denouncement_Date
                                        , a.L_3869_Protection_Flag
                                        , a.L_3869_Classification
                                        , a.Last_Legal_Action
                                        , a.Last_Legal_Action_Date
                                        , a.Legal_Process_Status
                                        , a.Base_Interest
                                        , a.Spread
                                        , a.N128
                                        , a.Penalty
                                        , a.Interest_Rate
                                        , a.ProductKey
                                        , dex.ProductKey expProductKey
                                        , substring(a.ProductKey,1,5) ProfitsProductCode
                                        , p.ProductDescription ProfitsProductDescription
                                        , substring(p.ProductKey,7,9) ProfitsABProductCode
                                        , sub.SubProductKey ABSubProductCode
                                        , sub.SubProductDescription ABSubProductDescription
                                        , dex.MaturityDate
                                        , dex.Principal 
                                        , dex.Interest  
                                        , dex.Commission + dex.Expense Expense
                                        , dex.DuePrincipal DelinquentPrincipal
                                        , dex.DueInterest + dex.PenaltyInterest DelinquentInterest
                                        , dex.DueExpense + dex.DueCommission DelinquentExpense
                                        , dex.DueUnrealizedInterest + dex.PenaltyUnrealizedInterest + dex.UnrealizedInterest URL_Interest
                                        , dex.Accrued
                                        , dex.DueAccrued DelinquentAccrued
                                        , CASE
                                              WHEN dex.InterestRateType = 'Fixed' and dex.InterestDuration != 999 THEN 'Fixed Periodic' 
                                              ELSE dex.InterestRateType
                                          END InterestRateType
                                        , dex.InterestCode
                                        , dex.InterestDescription
                                        , dex.baseCode BaseInterestCode
                                        , dex.baseDescription
                                        , dex.InterestValidityDate FixInterestStartDate
                                        , CASE 
                                              WHEN (dex.InterestRateType = 'Fixed' and dex.InterestDuration = 999) or (dex.InterestRateType = 'Variable') THEN NULL 
                                              ELSE dex.InterestFixedDate 
                                          END FixInterestEndDate
                                        , CASE 
                                              WHEN dex.InterestRateType = 'Variable' or (dex.InterestRateType = 'Fixed' and dex.InterestDuration = 999) THEN NULL 
                                              ELSE CASE 
                                                      WHEN dex.InterestDuration=999 or coalesce(dex.InterestDuration,0) = 0 THEN NULL 
                                                      ELSE CASE 
                                                              WHEN dex.InterestFrequency = 1 THEN concat(dex.InterestDuration, 'D') 
                                                              ELSE concat(dex.InterestDuration, 'M') 
                                                            END 
                                                  END 
                                          END InterestRateResetInterval
                                        , CASE
                                              WHEN dex.InterestCalcStopFlag = True THEN 'Yes' 
                                              ELSE 'No' 
                                          END InterestCalculationSuspended
                                FROM Loan a
                                INNER JOIN dexposure dex
                                ON a.Loan_ID = dex.ExposureKey
                                INNER JOIN Product p
                                ON dex.ProductKey = p.ProductKey
                                INNER JOIN subproduct sub
                                ON substring(a.Loan_ID,4,50) = sub.ExposureKey
                                LEFT JOIN dparty dpa
                                ON a.Borrower_ID = dpa.CDI
                            """)

In [0]:
if datatape_type ==  'Holiday':
        LoanRelations_sheet = spark.sql("""
                                        SELECT DISTINCT 
                                                MasterAccountNumber
                                                , SlaveAccountNumber
                                                , RelationType
                                        FROM MasterSlave a
                                        where exists (select 1 from Loan l where l.Loan_ID = a.SlaveAccountNumber)
                                        and exists (select 1 from Loan l where l.Loan_ID = a.MasterAccountNumber)
                                        """)

In [0]:
if datatape_type ==  'Holiday':
    OverdueRequests_sheet = spark.sql("""
                                        SELECT DISTINCT
                                                  a.Loan_ID
                                                , CASE 
                                                    WHEN r.REQUEST_TYPE = 1 THEN 'Installment' 
                                                    WHEN r.REQUEST_TYPE = 2 THEN 'Interest Request' 
                                                    WHEN r.REQUEST_TYPE = 3 THEN 'Loan Request' 
                                                    WHEN r.REQUEST_TYPE = 4 THEN 'ExpenseRequest'
                                                    WHEN r.REQUEST_TYPE = 5 THEN 'CapitalRequest'
										           END AS RequestType
                                                , CASE 
                                                    WHEN r.REQUEST_LOAN_STS = 1 THEN 'Normal' 
                                                    WHEN r.REQUEST_LOAN_STS = 2 then 'Overdue'
                                                    WHEN r.REQUEST_LOAN_STS = 3 then 'Define Delay' 
                                                    WHEN r.REQUEST_LOAN_STS = 4 then 'Writen Off'
                                                  END AS RequestLoansStatus
                                                ,  r.RQ_EXPIRE_DT  RequestExpiryDate
                                                , -r.RQ_CAPITAL_BAL CapitalBalance
                                                , -r.RQ_RL_NRM_INT_BAL InterestAmount
                                                , -r.RQ_RL_PNL_INT_BAL PenatlyInterestAmount
                                                , -r.RQ_EXP_BAL expenseAmount
                                                , -r.RQ_COM_BAL CommisionAmount
                                                , -r.RQ_URL_NRM_INT_BAL UnrealizedInterestAmount
                                                , -r.OV_ACCRUAL_AMN OverdueAccruals
                                        FROM Loan a
                                        INNER JOIN dexposure dex
                                        ON a.Loan_ID = dex.ExposureKey
                                        INNER JOIN profits_account pa
                                        ON pa.ACCOUNT_NUMBER = a.Loan_ID
                                        AND pa.PRFT_SYSTEM = 4
                                        INNER JOIN itstaging_loanrequest r
                                        ON r.FK_LOAN_ACCOUNTACC = pa.LNS_SN 
                                        AND r.FK_LOAN_ACCOUNTFK = pa.LNS_OPEN_UNIT
                                        AND r.FK0LOAN_ACCOUNTACC = pa.LNS_TYPE
                                        WHERE r.REQUEST_TYPE <> '3'
                                        and r.REQUEST_LOAN_STS in ('1','2')
                                        and r.REQUEST_STS = 1
                                    """)

In [0]:
if datatape_type ==  'Holiday':
    Relations_sheet = spark.sql("""
                                SELECT *
                                FROM Relations
                            """)

In [0]:
if datatape_type ==  'Holiday':
    Borrower_sheet = spark.sql("""
                                SELECT *
                                FROM Borrower
                            """)

In [0]:
if datatape_type ==  'Holiday':
    Collateral_sheet = spark.sql("""
                                SELECT DISTINCT
                                          ContractId
                                        , Collateral_code
                                        , Collateral_Status
                                        , CollateralStatusChangeDate
                                        , Collateral_Category
                                        , SubCategory
                                        , Collateral_Amount
                                        , Currency
                                        , Mortgage_Type
                                FROM Collateral
                            """)

In [0]:
if datatape_type ==  'Holiday':
    CollateralRE_sheet = spark.sql("""
                                    SELECT DISTINCT
                                              Collateral_code
                                            , CollateralRE_Id
                                            , CollateralRE_Status
                                            , `Ownership ID`
                                            , Property_ID
                                            , Lien_Order
                                            , Prenotation_Type
                                            , Registration_Date
                                            , `Αρ Δικαστικής Απόφασης`
                                            , `Υποθηκοφυλακείο`
                                            , `Σχόλια`
                                            , Common_Prenotation_Flag
                                    FROM CollateralRE
                                """)

In [0]:
if datatape_type ==  'Holiday':
    RE_sheet = spark.sql("""
                        SELECT DISTINCT
                                  Property_ID
                                , Property_Type
                                , Property_Sub_Type
                                , `Usage Type`
                                , RE_Asset_Description
                                , Prefecture
                                , Municipality
                                , Street_Name
                                , Postal_Code
                                , GPS_Coordinates
                                , Construction_Year
                                , `Land Area sqm`
                                , Main_Uses_sqm
                                , Main_Uses_Floor_Number
                                , Last_Appraisal_Value
                                , Last_Appraisal_Date
                                , `Appraisal Type`
                                , Marketability_Score
                                , RealEstateMunicipality `Δήμος`
                                , RealEstateDistrict `Περιφερειακή Ενότητα`
                                , RealEstateDivision `Περιφέρεια`
                        FROM RE
                        LEFT JOIN realestatebuilding
                        ON RE.Property_ID = realestatebuilding.RealEstateCepalKey
                        AND RE.Municipality <> realestatebuilding.RealEstateDistrict
                    """)

In [0]:
if datatape_type ==  'Holiday':
    REOwnership_sheet = spark.sql("""
                                SELECT DISTINCT
                                          PropertyOwnershipId Property_OwnershipId
                                        , Property_ID
                                        , CDI
                                        , VatNumber
                                        , PartyName
                                        , RealTitle_Id
                                        , `Ownership_%`
                                FROM ReOwnership
                            """)

In [0]:
if datatape_type ==  'Holiday':
    ThirdPartyLien_sheet = spark.sql("""
                                SELECT *
                                FROM ThirdPartyLien
                            """)

In [0]:
if datatape_type ==  'Holiday':
    CollateralNonRE_sheet = spark.sql("""
                                    SELECT DISTINCT
                                              Collateral_code
                                            , CollateralNRE_status
                                            , `Collateral_Type ID`
                                            , d.item_descr
                                            , PrimaryName
                                            , `Ενεχόμενος`
                                            , `Ενεχόμενος Name`
                                            , `Ημερομηνία εγγραφής εξασφάλισης`
                                            , `Σχόλια`
                                            , p.cepal_key `Linked Property ID`
                                            , case 
                                                when lp.collateral_ids is not null 
                                                    then 
                                                        case 
                                                            when charindex(',' ,lp.collateral_ids)>0 
                                                                then 
                                                                    case 
                                                                        when trim(lp.collateral_ids) = '29987, 31428' then 'NEP1597247 / NEP1611746' 
                                                                    end
                                                            else (select max(reference_key) from prcollateral prc where prc.id = lp.collateral_ids) 
                                                        end
                                              end `Linked Collaterals`
                                            , lp.auction_code `Auction Code`
                                            , hie.last_name `Επισπεύδων`
                                            , coalesce(cast(ppp.last_name as string), cast(lp.bidder_id as string)) `Υπερθεματιστής`
                                    FROM CollateralNonRE a
                                    LEFT JOIN prcollateral c
                                    ON c.reference_key = a.Collateral_code
                                    LEFT JOIN prcollateraldet d
                                    ON c.id = d.collateral_id
                                    LEFT JOIN prlinked_properties lp 
                                    ON lp.collateral_id = c.id 
                                    LEFT JOIN prproperty p 
                                    ON lp.property_id = p.id
                                    LEFT JOIN prperson ppp 
                                    ON lp.bidder_id = ppp.id
                                    LEFT JOIN datetape_hie hie
                                    ON hie.collateral_id = c.id
                                    where CollateralNRE_status = 'Ενεργή'
                                """)

In [0]:
if datatape_type ==  'Holiday':
    Payments_sheet = spark.sql("""
                                SELECT *
                                FROM Payments
                            """)

In [0]:
if datatape_type ==  'Holiday':
    PaymentPlan_sheet = spark.sql("""
                                SELECT *
                                FROM PaymentPlan
                            """)

## Type of Datatape

In [0]:
if datatape_type ==  'Holiday':
    dataframes = {
        "Loan Tape": LoanTape_sheet,
        "Loan Relations": LoanRelations_sheet,
        "Overdue Requests": OverdueRequests_sheet,
        "Relations": Relations_sheet,
        "Borrower": Borrower_sheet,
        "Collateral": Collateral_sheet,
        "CollateralRE": CollateralRE_sheet,
        "RE": RE_sheet,
        "RE Ownership": REOwnership_sheet,
        "3rdParty Lien": ThirdPartyLien_sheet,
        "Collateral NonRE": CollateralNonRE_sheet,
        "Payments": Payments_sheet,
        "PaymentPlan": PaymentPlan_sheet,
    }
elif datatape_type in  ('Classic','Neptune'):
    dataframes = {
        "Loan Tape": LoanTape_sheet,
        "Relations": Relations_sheet,
        "Representative": Representative_sheet,
        "Borrower": Borrower_sheet,
        "Collateral": Collateral_sheet,
        "CollateralRE": CollateralRE_sheet,
        "RE": RE_sheet,
        "RE Ownership": REOwnership_sheet,
        "3rdParty Lien": ThirdPartyLien_sheet,
        "Collateral NonRE": CollateralNonRE_sheet,
        "Payments": Payments_sheet,
        "PaymentPlan": PaymentPlan_sheet
    }

## Export to Volume

In [0]:
%pip install openpyxl

In [0]:
run_date_str = datetime.now().strftime("%Y%m%d")

#Define the output directory path
output_dir = f"{report_output}{run_date_str}/"

# Ensure the directory exists
os.makedirs(output_dir, exist_ok=True)

# Define the output file path
output_filename = f"{output_file_prefix}_{run_date.strftime('%Y%m%d')}.xlsx"
output_path = os.path.join(output_dir, output_filename)

In [0]:
output_path = '/Volumes/qa-cdh-delivery/internalreports/datatape/20261023/DataTape_20261022.xlsx'

In [0]:
# Prepare Excel buffer
output_buffer = BytesIO()
with pd.ExcelWriter(output_buffer, engine='openpyxl') as writer:
    for sheet_name, spark_df in dataframes.items():
        # Create temp folder for this sheet's CSV
        temp_file = os.path.join(output_dir, sheet_name)
        spark_df.coalesce(1).write.csv(temp_file, header=True, mode="overwrite")
        
        # Locate CSV file (skip metadata)
        csv_files = [f for f in os.listdir(temp_file) if f.endswith(".csv")]
        if csv_files:
            pandas_df = pd.read_csv(
                os.path.join(temp_file, csv_files[0]),
                engine="python",
                quotechar='"',
                escapechar='\\',
                on_bad_lines='warn'
            )
            pandas_df.to_excel(writer, sheet_name=sheet_name, index=False)
        
        # Clean up temporary CSV files
        for file in os.listdir(temp_file):
            os.remove(os.path.join(temp_file, file))

# Save Excel buffer to DBFS
with open(output_path, "wb") as f:
    f.write(output_buffer.getvalue())